# CASMI26 retrieval-only (v4): fixed, with built-in local experiments

Retrieval-only (no CheMM-R1 / ChemMRL). Defaults **reproduce the 0.150 pipeline** (`BASELINE_CFG`); a complete baseline `submission.csv` is written before anything else.

**What's different from the DeepSeek draft**
- Cell syntax error removed; formula enumeration dropped (any structure within 3 ppm is already inside the 10 ppm window, and its formula strings used a different element order from RDKit).
- Analogs are **in-pool only** (seed and candidate both inside the mass window; a different-mass structure cannot be the exact answer), so the bonus can only re-rank, never add wrong-mass guesses.
- Every idea is a switch in `BASELINE_CFG`-derived variants, evaluated on a local hold-out in two scenarios: *seen* (1-2 spectra hidden, siblings remain) and *unseen* (all spectra of the true structure hidden), weighted by the share of test molecules with a strong library match.
- The winner is applied only if its weighted MRR beats the baseline by `MIN_GAIN`; otherwise the baseline stays. Set `MANUAL_CFG` to force a configuration.
- Time guard: if less than 15 min remain, retrieval degrades to a fast mode instead of risking the deadline.

Outputs: `submission.csv`, `retrieval_debug.json` (includes the full validation table).

In [ ]:
!pip install -q --no-index --find-links /kaggle/input/datasets/eveliaveldrine/rdkit-wheels rdkit==2025.3.6

In [ ]:
import os, gc, io, re, json, time, math, random, warnings
from collections import defaultdict, Counter
from functools import lru_cache

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

from rdkit import RDLogger, DataStructs
from rdkit.Chem import (MolFromSmiles, MolToSmiles, MolToInchiKey,
                        rdMolDescriptors, rdFingerprintGenerator)
RDLogger.DisableLog("rdApp.*")

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ======================= TIME BUDGET =======================
NOTEBOOK_START = time.time()
HARD_DEADLINE_HOURS = 3.75
HARD_DEADLINE_SECONDS = HARD_DEADLINE_HOURS * 3600
def elapsed_min(): return (time.time() - NOTEBOOK_START) / 60
def remaining_s(): return HARD_DEADLINE_SECONDS - (time.time() - NOTEBOOK_START)

# ======================= PATHS =======================
COMP_DIRS = [
    "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra",
    "/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra",
    "./data", ".",
]
COMP_DIR = next((p for p in COMP_DIRS if os.path.exists(p)), COMP_DIRS[0])
TEST_PATH  = os.path.join(COMP_DIR, "test.parquet")
TRAIN_PATH = os.path.join(COMP_DIR, "train.parquet")
SAMPLE_SUB = os.path.join(COMP_DIR, "sample_submission.csv")
WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "."
SUBMISSION_PATH = os.path.join(WORKING_DIR, "submission.csv")
DEBUG_PATH = os.path.join(WORKING_DIR, "retrieval_debug.json")

# ======================= Library / retrieval constants =======================
MAX_STRUCTURES_FOR_LIBRARY = 600_000
MAX_SPECTRA_STORE = 12            # stored per structure; each experiment chooses how many to USE
MAX_QUERY_SPECTRA_SCORING = 8
PRIMARY_PPM = 10.0
FALLBACK_PPM = 30.0
FINAL_K = 25
MAX_QUERY_PEAKS = 128
PEAK_TOL = 0.02
STRONG_SPECTRAL = 0.70

# in-pool analog bonus: a candidate that is structurally close to a strongly matching
# library structure is more plausible. Both seed and candidate must be inside the mass
# window (a different-mass structure cannot be the exact answer).
ANALOG_SPECTRAL_MIN = 0.55
ANALOG_TANIMOTO_MIN = 0.55
ANALOG_TOP_K_SEEDS = 3

SOURCE_BONUS = {"train_retrieval": 0.0, "mass_fallback": -0.05, "nearest_mass": -0.05}

# ======================= EXPERIMENT CONFIGURATION =======================
# BASELINE_CFG reproduces the retrieval-only pipeline that scored 0.150.
#   spectra       : library spectra used per structure (first N of those stored)
#   cap           : max candidates taken from the mass window
#   w_*           : fusion weights; w_analog > 0 switches the analog bonus on
#   forced_strong : guarantee top slots to spectral matches >= STRONG_SPECTRAL
BASELINE_CFG = dict(spectra=3, cap=400, w_mass=0.15, w_spec=0.35, w_prior=0.05,
                    w_analog=0.0, forced_strong=True)

RUN_VALIDATION = True      # local hold-out experiments (~5-10 min) compare variants of the pipeline
AUTO_APPLY_BEST = True     # use the winner only if it beats the baseline by MIN_GAIN (weighted MRR)
MIN_GAIN = 0.02
N_VAL = 1500
NOISE_PPM = 1.5
MANUAL_CFG = None          # e.g. dict(BASELINE_CFG, spectra=12) to force a configuration

print("COMP_DIR:", COMP_DIR)
print("Baseline config:", BASELINE_CFG)

In [ ]:
ADDUCT_DELTAS = {
    "[M+H]+":          1.007276466621,
    "[M+NH4]+":        18.033823005,
    "[M-H2O+H]+":      -17.003288036,
    "[M-2H2O+H]+":     -35.013853101,
    "[M+Na]+":         22.989218,
    "[M+K]+":          38.963158,
    "[M-H]-":          -1.007276466621,
    "[M-H2O-H]-":      -19.0178415,
    "[M+CH2O2-H]-":    44.998201,
    "[M+Cl]-":         34.969402,
    "[M+FA-H]-":       44.998201,
    "[M+HCOO]-":       44.998201,
    "[M+Ac-H]-":       59.013851,
    "[M+CH3COO]-":     59.013851,
}

def neutral_mass_from_adduct(precursor_mz, adduct):
    adduct = str(adduct).strip()
    if adduct not in ADDUCT_DELTAS:
        raise ValueError(f"Unsupported CASMI adduct: {adduct}")
    return float(precursor_mz) - ADDUCT_DELTAS[adduct]

def safe_neutral_mass(precursor_mz, adduct):
    try:
        return neutral_mass_from_adduct(precursor_mz, adduct)
    except Exception:
        return np.nan

def ppm_error(mass, target):
    return abs(float(mass) - float(target)) / max(abs(float(target)), 1e-9) * 1e6

# Plain InChIKey connectivity block, cached. (The old tautomer-canonicalising key was
# the slowest step in the notebook -- it ran for every candidate of every molecule --
# and it did not match the InChIKey14 the scorer computes from the submitted SMILES.)
@lru_cache(maxsize=1_000_000)
def ikey14(smi):
    mol = MolFromSmiles(smi) if isinstance(smi, str) else None
    if mol is None:
        return None
    try:
        k = MolToInchiKey(mol)
        return k.split("-")[0] if k else None
    except Exception:
        return None

def structure_prior_from_mol(mol):
    heavy = mol.GetNumHeavyAtoms()
    rings = rdMolDescriptors.CalcNumRings(mol)
    hetero = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() in (7, 8, 15, 16, 17))
    return float(0.002 * min(heavy, 80) + 0.01 * min(rings, 8) + 0.003 * min(hetero, 12))

print("Supported adducts:", len(ADDUCT_DELTAS))

In [ ]:
test_df = pd.read_parquet(TEST_PATH)
required = {"molecule_id", "precursor_mz", "adduct",
            "ms2_mzs", "ms2_normalized_intensities"}
missing = required - set(test_df.columns)
if missing:
    raise RuntimeError(f"Missing required columns: {missing}")

test_df["neutral_mass"] = [
    safe_neutral_mass(mz, ad)
    for mz, ad in zip(test_df["precursor_mz"], test_df["adduct"])
]

print(f"Test spectra: {len(test_df):,}")
print(f"Test molecules: {test_df.molecule_id.nunique():,}")
print(f"Unsupported adduct rows: {test_df['neutral_mass'].isna().sum()}")

molecule_meta = []
for mid, g in test_df.groupby("molecule_id", sort=False):
    masses = g["neutral_mass"].dropna().to_numpy(dtype=np.float64)
    if len(masses) == 0:
        continue
    med = float(np.median(masses))
    molecule_meta.append({
        "molecule_id": mid,
        "target_mass": med,
        "mass_spread_ppm": float((masses.max() - masses.min()) / max(med, 1e-9) * 1e6)
                           if len(masses) > 1 else 0.0,
        "n_spectra": len(g),
    })
molecule_meta = pd.DataFrame(molecule_meta)
target_mass_by_mid = dict(zip(molecule_meta["molecule_id"], molecule_meta["target_mass"]))
print(f"Molecules prepared: {len(molecule_meta)}")
print(molecule_meta["mass_spread_ppm"].describe()[["mean", "50%", "max"]])

In [ ]:
_EMPTY = (np.empty(0, np.float32), np.empty(0, np.float32))

def select_peaks(mzs, intensities, max_peaks=MAX_QUERY_PEAKS, noise_floor=0.005):
    """Clean a spectrum: finite/positive peaks, base-peak normalised, noise filtered,
    top-N by intensity, and ALWAYS sorted by m/z (the matcher needs sorted m/z; the old
    code only preserved input order)."""
    if mzs is None or intensities is None:
        return _EMPTY
    mzs = np.asarray(mzs, dtype=np.float64)
    ints = np.asarray(intensities, dtype=np.float64)
    n = min(mzs.size, ints.size)
    if n == 0:
        return _EMPTY
    mzs, ints = mzs[:n], ints[:n]
    ok = np.isfinite(mzs) & np.isfinite(ints) & (ints > 0)
    mzs, ints = mzs[ok], ints[ok]
    if ints.size == 0:
        return _EMPTY
    ints = ints / ints.max()
    keep = ints >= noise_floor
    mzs, ints = mzs[keep], ints[keep]
    if mzs.size > max_peaks:
        top = np.argsort(-ints, kind="stable")[:max_peaks]
        mzs, ints = mzs[top], ints[top]
    order = np.argsort(mzs, kind="stable")
    return mzs[order].astype(np.float32), ints[order].astype(np.float32)

def to_weights(ints):
    """sqrt-intensity, L2-normalised (down-weights dominant peaks, standard for MS/MS cosine)."""
    w = np.sqrt(ints.astype(np.float64))
    n = np.linalg.norm(w)
    return (w / n).astype(np.float32) if n > 0 else w.astype(np.float32)

def _nearest(l_mz, x, tol):
    """For every x find the nearest peak in sorted l_mz; return (index, within_tol)."""
    n = l_mz.shape[0]
    pos = np.searchsorted(l_mz, x)
    lo = np.clip(pos - 1, 0, n - 1)
    hi = np.clip(pos, 0, n - 1)
    dlo = np.abs(l_mz[lo] - x)
    dhi = np.abs(l_mz[hi] - x)
    use_hi = dhi < dlo
    return np.where(use_hi, hi, lo), np.minimum(dlo, dhi) <= tol

def _score_parts(qi_parts, lj_parts, pr_parts):
    if not qi_parts:
        return 0.0
    qi = np.concatenate(qi_parts); lj = np.concatenate(lj_parts); pr = np.concatenate(pr_parts)
    if len(qi_parts) > 1 or np.unique(lj).size < lj.size:
        # greedy one-to-one assignment: best products first, each peak used once
        o = np.argsort(-pr, kind="stable"); qi, lj, pr = qi[o], lj[o], pr[o]
        _, f = np.unique(qi, return_index=True); lj, pr = lj[f], pr[f]
        o = np.argsort(-pr, kind="stable"); lj, pr = lj[o], pr[o]
        _, f = np.unique(lj, return_index=True); pr = pr[f]
    return float(min(pr.sum(), 1.0))

def spectral_similarity(q_mz, q_w, l_mz, l_w, shift=0.0, tol=PEAK_TOL, with_plain=False):
    """(Modified) cosine between two sorted spectra, vectorised, one-to-one matching.
    shift = precursor difference; peaks may match directly or shifted (modified cosine).
    with_plain=True returns (modified, plain) from a single matching pass."""
    if q_mz.size == 0 or l_mz.size == 0:
        return (0.0, 0.0) if with_plain else 0.0
    qi_parts, lj_parts, pr_parts = [], [], []
    j, ok = _nearest(l_mz, q_mz, tol)
    if ok.any():
        qi = np.flatnonzero(ok)
        qi_parts.append(qi); lj_parts.append(j[ok]); pr_parts.append(q_w[qi] * l_w[j[ok]])
    n_direct = len(qi_parts)
    if abs(shift) > tol:
        j2, ok2 = _nearest(l_mz, q_mz - shift, tol)
        if ok2.any():
            qi = np.flatnonzero(ok2)
            qi_parts.append(qi); lj_parts.append(j2[ok2]); pr_parts.append(q_w[qi] * l_w[j2[ok2]])
    mod = _score_parts(qi_parts, lj_parts, pr_parts)
    if not with_plain:
        return mod
    plain = mod if len(qi_parts) == n_direct else _score_parts(qi_parts[:n_direct], lj_parts[:n_direct], pr_parts[:n_direct])
    return mod, plain

In [ ]:
def prepare_query_spectra(group):
    prepared = []
    for mz_arr, int_arr, pmz, nm in zip(group["ms2_mzs"], group["ms2_normalized_intensities"],
                                        group["precursor_mz"], group["neutral_mass"]):
        if not np.isfinite(nm):
            continue
        mz, it = select_peaks(mz_arr, int_arr)
        if len(mz) < 3:
            continue
        prepared.append({"mz": mz, "w": to_weights(it), "prec": float(pmz)})
    prepared.sort(key=lambda x: len(x["mz"]), reverse=True)
    return prepared

query_cache = {}
for mid, g in tqdm(test_df.groupby("molecule_id", sort=False), desc="Preparing queries"):
    query_cache[mid] = prepare_query_spectra(g)
print("Prepared queries for", len(query_cache), "molecules |",
      "without usable spectra:", sum(1 for v in query_cache.values() if not v))

In [ ]:
# Stream the training parquet in batches (no giant DataFrame), keep up to
# MAX_SPECTRA_STORE spectra per structure, store spectra as compact float32 arrays.
print("Building spectral library from train...")
t0 = time.time()
pf = pq.ParquetFile(TRAIN_PATH)
train_cols = set(pf.schema_arrow.names)
cols = ["inchikey14", "normalized_smiles", "ms2_mzs", "ms2_normalized_intensities"]
HAS_PREC = "precursor_mz" in train_cols
if HAS_PREC:
    cols.append("precursor_mz")      # enables modified-cosine matching across adducts
print(f"Train rows: {pf.metadata.num_rows:,} | precursor_mz available: {HAS_PREC}")

struct_index = {}                    # inchikey14 -> structure idx (-1 = unparsable)
lib_smiles, lib_ikey, lib_mass_l, lib_prior_l, struct_specs = [], [], [], [], []
spec_mz, spec_w, spec_prec = [], [], []

pbar = tqdm(total=pf.metadata.num_rows, desc="Indexing train")
for batch in pf.iter_batches(batch_size=20_000, columns=cols):
    df = batch.to_pandas()
    pbar.update(len(df))
    prec_col = df["precursor_mz"].to_numpy() if HAS_PREC else None
    for r, (ikey, smi, mzs, ints) in enumerate(zip(df["inchikey14"], df["normalized_smiles"],
                                                   df["ms2_mzs"], df["ms2_normalized_intensities"])):
        if not isinstance(smi, str) or not isinstance(ikey, str) or not ikey:
            continue
        idx = struct_index.get(ikey)
        if idx is None:
            if len(lib_smiles) >= MAX_STRUCTURES_FOR_LIBRARY:
                continue
            mol = MolFromSmiles(smi)
            if mol is None:
                struct_index[ikey] = -1
                continue
            try:
                mw = rdMolDescriptors.CalcExactMolWt(mol)
                prior = structure_prior_from_mol(mol)
            except Exception:
                struct_index[ikey] = -1
                continue
            idx = len(lib_smiles)
            struct_index[ikey] = idx
            lib_smiles.append(smi); lib_ikey.append(ikey)
            lib_mass_l.append(mw); lib_prior_l.append(prior); struct_specs.append([])
        elif idx < 0:
            continue
        if len(struct_specs[idx]) >= MAX_SPECTRA_STORE:
            continue
        mz, it = select_peaks(mzs, ints)
        if len(mz) < 3:
            continue
        sid = len(spec_mz)
        spec_mz.append(mz); spec_w.append(to_weights(it))
        spec_prec.append(float(prec_col[r]) if HAS_PREC else float("nan"))
        struct_specs[idx].append(sid)
    del df
pbar.close()
del struct_index; gc.collect()

if not lib_smiles:
    raise RuntimeError("Library is empty - check TRAIN_PATH / column names")

# sort structures by mass for binary-search lookups
order = np.argsort(np.asarray(lib_mass_l, dtype=np.float64), kind="stable")
lib_mass  = np.asarray(lib_mass_l, dtype=np.float64)[order]
lib_prior = np.asarray(lib_prior_l, dtype=np.float32)[order]
lib_smiles = [lib_smiles[i] for i in order]
lib_ikey   = [lib_ikey[i] for i in order]
struct_specs = [struct_specs[i] for i in order]
spec_prec = np.asarray(spec_prec, dtype=np.float64)
del lib_mass_l, lib_prior_l, order; gc.collect()

n_with_spec = sum(1 for s in struct_specs if s)
print(f"Library: {len(lib_smiles):,} structures | {len(spec_mz):,} spectra "
      f"({n_with_spec:,} structures with >=1 spectrum) | {(time.time()-t0)/60:.1f} min")
print(f"Elapsed: {elapsed_min():.1f} min")

In [ ]:
EXCLUDE_SIDS = set()        # spectra hidden from matching (validation only)

def candidate_spectral_score(idx, qspecs, max_spectra):
    """Best (modified) cosine between any of the first `max_spectra` library spectra of this
    structure and any query spectrum."""
    best = 0.0
    for sid in struct_specs[idx][:max_spectra]:
        if sid in EXCLUDE_SIDS:
            continue
        lmz, lw, lp = spec_mz[sid], spec_w[sid], spec_prec[sid]
        for q in qspecs:
            shift = (q["prec"] - lp) if (np.isfinite(lp) and np.isfinite(q["prec"])) else 0.0
            s = spectral_similarity(q["mz"], q["w"], lmz, lw, shift)
            if s > best:
                best = s
    return best

def _window(target_mass, ppm):
    d = target_mass * ppm * 1e-6
    lo = int(np.searchsorted(lib_mass, target_mass - d, side="left"))
    hi = int(np.searchsorted(lib_mass, target_mass + d, side="right"))
    return lo, hi

def _closest(lo, hi, target_mass, limit):
    idx = np.arange(lo, hi)
    if idx.size == 0:
        return idx
    return idx[np.argsort(np.abs(lib_mass[idx] - target_mass), kind="stable")[:limit]]

def retrieve_candidates(target_mass, qspecs, cap, max_spectra):
    """Mass-window candidates scored by spectral similarity. Widens to FALLBACK_PPM, then to the
    nearest masses, so the 25 guess slots are never wasted."""
    qs = qspecs[:MAX_QUERY_SPECTRA_SCORING]
    chosen = {}
    lo, hi = _window(target_mass, PRIMARY_PPM)
    for i in _closest(lo, hi, target_mass, cap):
        chosen[int(i)] = "train_retrieval"
    if len(chosen) < FINAL_K:
        lo, hi = _window(target_mass, FALLBACK_PPM)
        for i in _closest(lo, hi, target_mass, cap):
            if len(chosen) >= cap:
                break
            chosen.setdefault(int(i), "mass_fallback")
    if not chosen:
        pos = int(np.searchsorted(lib_mass, target_mass))
        lo, hi = max(0, pos - FINAL_K), min(len(lib_mass), pos + FINAL_K)
        for i in _closest(lo, hi, target_mass, FINAL_K):
            chosen[int(i)] = "nearest_mass"
    out = []
    for i, src in chosen.items():
        mw = float(lib_mass[i])
        out.append({"idx": i, "smiles": lib_smiles[i], "mass": mw, "ppm": ppm_error(mw, target_mass),
                    "spectral_score": candidate_spectral_score(i, qs, max_spectra) if qs else 0.0,
                    "prior_score": float(lib_prior[i]), "source": src})
    return out

# ---------------- in-pool analog bonus ----------------
_fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
_EMPTY_FP = DataStructs.ExplicitBitVect(2048)
_fp_cache = {}
def lib_fp(i):
    f = _fp_cache.get(i)
    if f is None:
        m = MolFromSmiles(lib_smiles[i])
        f = _fp_gen.GetFingerprint(m) if m is not None else _EMPTY_FP
        _fp_cache[i] = f
    return f

def analog_scores(pool):
    """analog(c) = max over the top-K spectral seeds s != c of  spectral(s) * Tanimoto(c, s)  (if >= threshold)."""
    out = np.zeros(len(pool), dtype=np.float64)
    seeds = [j for j in sorted(range(len(pool)), key=lambda j: -pool[j]["spectral_score"])
             if pool[j]["spectral_score"] >= ANALOG_SPECTRAL_MIN][:ANALOG_TOP_K_SEEDS]
    if not seeds:
        return out
    fps = [lib_fp(e["idx"]) for e in pool]
    for sj in seeds:
        sims = np.asarray(DataStructs.BulkTanimotoSimilarity(fps[sj], fps), dtype=np.float64)
        sims[sj] = 0.0
        sims[sims < ANALOG_TANIMOTO_MIN] = 0.0
        out = np.maximum(out, pool[sj]["spectral_score"] * sims)
    return out

# ---------------- fusion ----------------
def fuse(target_mass, pool, cfg):
    ent = [dict(e) for e in pool]
    an = analog_scores(ent) if cfg["w_analog"] > 0 else np.zeros(len(ent))
    for e, a in zip(ent, an):
        e["analog_score"] = float(a)
        e["fusion_score"] = (cfg["w_mass"] * math.exp(-e["ppm"] / 5.0) + cfg["w_spec"] * e["spectral_score"]
                             + cfg["w_prior"] * e["prior_score"] + cfg["w_analog"] * float(a)
                             + SOURCE_BONUS.get(e["source"], 0.0))
    ranked = sorted(ent, key=lambda x: x["fusion_score"], reverse=True)
    if cfg["forced_strong"]:
        forced = [x for x in sorted(ent, key=lambda x: x["spectral_score"], reverse=True)[:2]
                  if x["spectral_score"] >= STRONG_SPECTRAL and x["ppm"] <= FALLBACK_PPM]
        forced.sort(key=lambda x: x["fusion_score"], reverse=True)
        fid = {id(x) for x in forced}
        ranked = forced + [x for x in ranked if id(x) not in fid]
    selected, seen = [], set()
    for x in ranked:
        k = ikey14(x["smiles"]) or x["smiles"]
        if k in seen:
            continue
        seen.add(k); x["key"] = k; selected.append(x)
        if len(selected) >= FINAL_K:
            break
    return selected

def write_submission(final_records, path=SUBMISSION_PATH):
    sample_sub = pd.read_csv(SAMPLE_SUB)
    id_col = "molecule_id"
    target_col = "smiles" if "smiles" in sample_sub.columns else next(c for c in sample_sub.columns if c != id_col)
    rows = []
    for mid in sample_sub[id_col]:
        guesses = [c["smiles"] for c in final_records.get(mid, [])][:FINAL_K]
        rows.append(";".join(guesses) if guesses else "CCO")
    sample_sub[target_col] = rows
    sample_sub.to_csv(path, index=False)
    return sample_sub, target_col

print("Retrieval + fusion ready")

In [ ]:
_pool_cache = {}
def get_test_pools(cfg):
    key = (cfg["spectra"], cfg["cap"])
    if key in _pool_cache:
        return _pool_cache[key]
    EXCLUDE_SIDS.clear()
    pools = {}
    for mid, tm in tqdm(target_mass_by_mid.items(), desc=f"Test retrieval {key}"):
        if remaining_s() < 900:      # time guard: finish quickly rather than risk the deadline
            pools[mid] = retrieve_candidates(float(tm), query_cache[mid], 60, 2)
        else:
            pools[mid] = retrieve_candidates(float(tm), query_cache[mid], cfg["cap"], cfg["spectra"])
    _pool_cache[key] = pools
    return pools

def run_final(cfg):
    pools = get_test_pools(cfg)
    return {mid: fuse(float(target_mass_by_mid[mid]), pool, cfg) for mid, pool in tqdm(pools.items(), desc="Fusion")}

t0 = time.time()
base_pools = get_test_pools(BASELINE_CFG)
best_spec = {m: max((c["spectral_score"] for c in p), default=0.0) for m, p in base_pools.items()}
vals = np.array(list(best_spec.values()))
SEEN_FRACTION = float((vals >= STRONG_SPECTRAL).mean())
print(f"Retrieval {(time.time()-t0)/60:.1f} min | best spectral score quantiles (10/50/90%): "
      f"{np.round(np.quantile(vals, [0.1, 0.5, 0.9]), 3)}")
print(f"Test molecules with a strong library match (>= {STRONG_SPECTRAL}): {(vals >= STRONG_SPECTRAL).sum()} / {len(vals)} "
      f"-> SEEN_FRACTION = {SEEN_FRACTION:.2f}")

# Safety net: a complete baseline submission exists from this point on.
final_records = run_final(BASELINE_CFG)
write_submission(final_records)
print("Baseline submission written:", SUBMISSION_PATH, "| elapsed", f"{elapsed_min():.1f} min")

In [ ]:
VALIDATION_RESULTS = {}
CHOSEN_CFG = dict(BASELINE_CFG)

def _metrics(ranks):
    r = np.asarray(ranks, dtype=np.float64)
    return {"top1": float((r < 1).mean()), "top5": float((r < 5).mean()),
            "top25": float((r < 25).mean()), "MRR": float(np.mean(1.0 / (r + 1.0)))}

if RUN_VALIDATION and remaining_s() > 3600:
    t0 = time.time()
    rng = np.random.default_rng(SEED)
    eligible = [i for i, s in enumerate(struct_specs) if len(s) >= 2]
    chosen_idx = rng.choice(eligible, size=min(N_VAL, len(eligible)), replace=False)
    val = []
    for i in chosen_idx:
        sids = struct_specs[i]
        first = sids[:3]                                  # hidden spectra come from the first 3 so
        k = 2 if len(first) >= 3 else 1                   # every variant (3 or 12 spectra) can see siblings
        held = [int(x) for x in rng.choice(first, size=k, replace=False)]
        qs = sorted([{"mz": spec_mz[s], "w": spec_w[s], "prec": float(spec_prec[s])} for s in held],
                    key=lambda x: -len(x["mz"]))
        tm = float(lib_mass[i]) * (1.0 + rng.normal(0.0, NOISE_PPM) * 1e-6)
        val.append({"idx": int(i), "held": held, "all": [int(s) for s in sids], "qs": qs, "tm": tm})
    print(f"Validation: {len(val)} molecules | seen-fraction weight {SEEN_FRACTION:.2f}")

    def val_pools(scenario, spectra):
        EXCLUDE_SIDS.clear()
        if scenario == "seen":                            # hide 1-2 spectra; siblings remain
            EXCLUDE_SIDS.update(s for v in val for s in v["held"])
        else:                                             # "unseen": hide ALL spectra of the true structure
            EXCLUDE_SIDS.update(s for v in val for s in v["all"])
        pools = [retrieve_candidates(v["tm"], v["qs"], BASELINE_CFG["cap"], spectra)
                 for v in tqdm(val, desc=f"val pools {scenario}/{spectra}")]
        EXCLUDE_SIDS.clear()
        return pools

    pool_store = {}
    def evaluate(cfg):
        out = {}
        for scen in ("seen", "unseen"):
            key = (scen, cfg["spectra"])
            if key not in pool_store:
                pool_store[key] = val_pools(*key)
            ranks, in_pool = [], []
            for v, pool in zip(val, pool_store[key]):
                fused = fuse(v["tm"], pool, cfg)
                pos = [j for j, e in enumerate(fused) if e["idx"] == v["idx"]]
                ranks.append(pos[0] if pos else 10**6)
                in_pool.append(any(e["idx"] == v["idx"] for e in pool))
            m = _metrics(ranks); m["in_pool"] = float(np.mean(in_pool))
            out[scen] = m
        out["weighted_MRR"] = SEEN_FRACTION * out["seen"]["MRR"] + (1 - SEEN_FRACTION) * out["unseen"]["MRR"]
        return out

    B = BASELINE_CFG
    singles = {
        "12 spectra per structure":   dict(B, spectra=12),
        "mass weight 0.02":           dict(B, w_mass=0.02),
        "spectral-first (0.02/0.60)": dict(B, w_mass=0.02, w_spec=0.60),
        "in-pool analog bonus 0.20":  dict(B, w_analog=0.20),
        "no forced strong slots":     dict(B, forced_strong=False),
    }
    VALIDATION_RESULTS["baseline"] = (dict(B), evaluate(B))
    for name, cfg in singles.items():
        VALIDATION_RESULTS[name] = (cfg, evaluate(cfg))
    base_w = VALIDATION_RESULTS["baseline"][1]["weighted_MRR"]
    winners = [n for n in singles if VALIDATION_RESULTS[n][1]["weighted_MRR"] > base_w + 0.005]
    if len(winners) >= 2:                                 # combine everything that helped on its own
        combo = dict(B)
        for n in winners:
            for k, v in singles[n].items():
                if v != B[k]:
                    combo[k] = v
        VALIDATION_RESULTS["combo of winners: " + ", ".join(winners)] = (combo, evaluate(combo))

    rows = []
    for name, (cfg, r) in VALIDATION_RESULTS.items():
        rows.append({"variant": name[:60],
                     "seen top1": round(r["seen"]["top1"] * 100, 1), "seen top25": round(r["seen"]["top25"] * 100, 1),
                     "unseen top25": round(r["unseen"]["top25"] * 100, 1),
                     "seen MRR": round(r["seen"]["MRR"], 3), "unseen MRR": round(r["unseen"]["MRR"], 3),
                     "weighted MRR": round(r["weighted_MRR"], 3)})
    print(pd.DataFrame(rows).to_string(index=False))
    r0 = VALIDATION_RESULTS["baseline"][1]
    print(f"\nTruth inside candidate pool (cap {B['cap']}): seen {r0['seen']['in_pool']*100:.1f}% | unseen {r0['unseen']['in_pool']*100:.1f}%")

    best_name = max(VALIDATION_RESULTS, key=lambda n: VALIDATION_RESULTS[n][1]["weighted_MRR"])
    gain = VALIDATION_RESULTS[best_name][1]["weighted_MRR"] - base_w
    print(f"Best: '{best_name}' (weighted MRR {gain:+.3f} vs baseline; required {MIN_GAIN:+.3f})")
    if AUTO_APPLY_BEST and best_name != "baseline" and gain >= MIN_GAIN:
        CHOSEN_CFG = dict(VALIDATION_RESULTS[best_name][0])
        print("-> applying:", CHOSEN_CFG)
    else:
        print("-> keeping the baseline configuration")
    print(f"Validation took {(time.time()-t0)/60:.1f} min | elapsed {elapsed_min():.1f} min")
else:
    print("Validation skipped.")
if MANUAL_CFG:
    CHOSEN_CFG = dict(MANUAL_CFG)
    print("MANUAL_CFG override:", CHOSEN_CFG)

In [ ]:
if CHOSEN_CFG != BASELINE_CFG:
    final_records = run_final(CHOSEN_CFG)
    print("Final records built with:", CHOSEN_CFG)
else:
    print("Chosen config == baseline; reusing baseline records")

sample_sub, target_col = write_submission(final_records)
print("Submission:", SUBMISSION_PATH, "| rows:", len(sample_sub))
print("Guesses per row:", sample_sub[target_col].str.split(";").map(len).describe()[["min", "50%", "max"]].to_dict())
invalid = total = 0
for row in sample_sub[target_col]:
    for smi in row.split(";"):
        total += 1
        invalid += MolFromSmiles(smi) is None
print(f"Invalid SMILES: {invalid}/{total}")
assert len(sample_sub) > 0 and invalid == 0, "Submission validation failed"

elapsed = time.time() - NOTEBOOK_START
src = Counter(c["source"] for v in final_records.values() for c in v)
debug = {
    "runtime_minutes": round(elapsed / 60, 1), "molecules": len(target_mass_by_mid),
    "library_structures": len(lib_smiles), "library_spectra": len(spec_mz),
    "seen_fraction": SEEN_FRACTION, "chosen_cfg": CHOSEN_CFG, "baseline_cfg": BASELINE_CFG,
    "final_source_counts": dict(src),
    "final_with_spectral_score": sum(c["spectral_score"] > 0 for v in final_records.values() for c in v),
    "final_with_analog_score": sum(c.get("analog_score", 0) > 0 for v in final_records.values() for c in v),
    "validation": {n: {"cfg": c, "weighted_MRR": r["weighted_MRR"], "seen": r["seen"], "unseen": r["unseen"]}
                   for n, (c, r) in VALIDATION_RESULTS.items()},
}
with open(DEBUG_PATH, "w") as f:
    json.dump(debug, f, indent=2, default=str)
print(json.dumps({k: v for k, v in debug.items() if k != "validation"}, indent=2, default=str))
print(f"\nDone. Wall time: {elapsed/60:.1f} min (deadline {HARD_DEADLINE_HOURS*60:.0f} min)")